In [ ]:
import os
import time
import warnings

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hyperparameter tuning -- TabPFN

Optuna tuning of TabPFN on the feature datasets, with the same validation
protocol, objective and trial budget as `07a_tuning_features_ml.ipynb`
and `07d_tuning_timeseries_dl.ipynb`.

TabPFN needs its own environment (`tabpfn`), so it is tuned here rather
than in `07b_tuning_features_dl.ipynb` (GANDALF and Category Embedding,
env `torch_tabular`). Outputs are written to
`behaviour_features_dl_tabpfn_tuned_*.csv` (see "Save outputs").

## Validation protocol

Uses the 5 stratified-group CV folds in `indices_cv_fold_lookup.csv`
(built by `04_build_cv_fold_lookup.ipynb`). For each test fold i:

- Inner validation: the next `N_INNER_FOLDS` = 2 folds, (i+1) mod 5 and
  (i+2) mod 5. For each, TabPFN is trained on the remaining 3 folds and
  scored on that fold; a trial's score is the mean validation `f1_macro`
  over the 2 inner folds.
- Optuna runs `N_TRIALS` = 50 trials (TPE sampler, `seed=RANDOM_STATE`,
  `multivariate=True`).
- The best setting is retrained on all 4 non-test folds and evaluated once
  on the test fold.

## Search space

`balance_probabilities` is fixed to `use_class_weight` (True), as in
`05c_feature_classification_dl_tabpfn.ipynb`. Tuned parameters:

- `n_estimators`: integer 4-32, log-scaled (default 4). Ensemble size.
- `outlier_removal_std`: float 3.0-20.0 (default 12.0, TabPFN's "auto"
  value for classification), passed as
  `inference_config={"OUTLIER_REMOVAL_STD": ...}`. Sets how strongly
  extreme feature values are clipped before they reach the pretrained
  network.

## Default control

`TABPFN_DEFAULT_PARAMS` (`n_estimators=4, outlier_removal_std=12.0`, the
configuration used in `05c_feature_classification_dl_tabpfn.ipynb`) is
used twice per test fold:

- Validation side: it is enqueued as the first of the `N_TRIALS` trials,
  and its score is saved as `default_val_f1_macro` next to the winner's
  `val_f1_macro`.
- Test side: it is retrained on the same 4 non-test folds and evaluated on
  the same test fold as the tuned winner, with the same code path and
  seed, so tuned vs default is like-for-like. These rows
  (`model="TabPFN_default"`) are saved to separate
  `..._tuned_default_control_*.csv` files, so the `..._tuned_metrics.csv`
  / `..._tuned_confusion.csv` files read by
  `12b_publication_plots_fig3.R` contain only tuned rows.

## Scope

Burst_1/uncorrected only (`DATASETS`), behaviour target only.


## Load the main CV fold lookup and set tuning constants

In [ ]:
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
DATASETS = [
    ("burst_1", "uncorrected"),
]
N_TRIALS = 50
RANDOM_STATE = 42
# Number of inner validation folds whose f1_macro is averaged per Optuna
# trial (folds (i+1) and (i+2) mod 5 for outer test fold i).
N_INNER_FOLDS = 2
# Default control: TabPFN's own defaults, as used in
# 05c_feature_classification_dl_tabpfn.ipynb (n_estimators=4,
# OUTLIER_REMOVAL_STD "auto" = 12.0 for classification). Enqueued as the
# first search trial and also retrained on each test fold.
TABPFN_DEFAULT_PARAMS = {"n_estimators": 4, "outlier_removal_std": 12.0}

fold_lookup_all = pd.read_csv(FOLD_LOOKUP_PATH)
print(
    f"Loaded fold lookup: {len(fold_lookup_all)} rows, "
    f"bursts={sorted(fold_lookup_all['burst'].unique())}"
)

## TabPFN Optuna search and final evaluation for one test fold


In [ ]:
def make_tabpfn_objective(cv_folds, use_class_weight=True):
    """Build an Optuna objective that returns the mean f1_macro across the
    inner (train, val) splits in cv_folds.

    Args:
        cv_folds (list): list of (X_train, y_train, X_val, y_val) tuples,
            one per inner fold.
    """

    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 4, 32, log=True)
        outlier_removal_std = trial.suggest_float("outlier_removal_std", 3.0, 20.0)

        scores = []
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for X_train, y_train, X_val, y_val in cv_folds:
                model = TabPFNClassifier(
                    n_estimators=n_estimators,
                    balance_probabilities=use_class_weight,
                    inference_config={"OUTLIER_REMOVAL_STD": outlier_removal_std},
                    random_state=RANDOM_STATE,
                )
                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)
                scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))
        return float(np.mean(scores))

    return objective


def retrain_and_evaluate_tabpfn(
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    params,
    class_names,
    use_class_weight=True,
):
    """Fit TabPFN with `params` on X_train_val/y_train_val and evaluate
    once on X_test/y_test. Returns (metrics_row, confusion_rows);
    confusion_rows has no "model" key (the caller adds it).

    Used for both the tuned winner and TABPFN_DEFAULT_PARAMS, so both go
    through the same code path and seed.
    """
    n_classes = len(class_names)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model = TabPFNClassifier(
            n_estimators=params["n_estimators"],
            balance_probabilities=use_class_weight,
            inference_config={"OUTLIER_REMOVAL_STD": params["outlier_removal_std"]},
            random_state=RANDOM_STATE,
        )
        model.fit(X_train_val, y_train_val)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        try:
            roc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        metrics_row = {
            "n_estimators": params["n_estimators"],
            "outlier_removal_std": params["outlier_removal_std"],
            "roc_auc": roc,
            "accuracy": model.score(X_test, y_test),
            "balanced_acc": balanced_accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "precision": precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            ),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "precision_macro": precision_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_test, y_pred, average="macro", zero_division=0
            ),
            "time": time.time() - t0,
        }

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
        confusion_rows = [
            {
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(n_classes)
            for j in range(n_classes)
        ]

    return metrics_row, confusion_rows


def tune_and_evaluate_tabpfn_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one test fold for TabPFN: Optuna search scored by the mean
    inner-validation f1_macro (make_tabpfn_objective), then retrain the
    best setting on all non-test folds and evaluate once on the test fold.

    TABPFN_DEFAULT_PARAMS is enqueued as the first trial; its validation
    score is returned as default_val_f1_macro. It is also retrained and
    evaluated on the same split as the winner (model="TabPFN_default").

    One Optuna study is created per test fold, named
    "{burst}_{correction_type}_testfold{test_fold}", and returned for the
    "Inspect best study per dataset" cell.

    Args:
        cv_folds (list): list of (X_train, y_train, X_val, y_val) tuples
            for the inner validation search (see make_tabpfn_objective).
        X_train_val, y_train_val: all non-test-fold data combined, for the
            final retrain with the winning hyperparameters.
        X_test, y_test: the held-out test fold, touched only once, after
            hyperparameters are already fixed.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). metrics_df/confusion_df hold the
            tuned winner (model="TabPFN"); default_metrics_df/
            default_confusion_df hold the default control
            (model="TabPFN_default"), kept separate because they are saved
            to separate files.
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )
    study.enqueue_trial(TABPFN_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_tabpfn_objective(cv_folds, use_class_weight=use_class_weight),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    # Validation score of the enqueued default trial, found by matching
    # params.
    default_trial = next(
        (t for t in study.trials if t.params == TABPFN_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    tuned_row, tuned_conf_rows = retrain_and_evaluate_tabpfn(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        best_params,
        class_names,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "TabPFN",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "TabPFN", **row} for row in tuned_conf_rows]

    # Default control, retrained and evaluated on the same split.
    default_row, default_conf_rows = retrain_and_evaluate_tabpfn(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        TABPFN_DEFAULT_PARAMS,
        class_names,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "TabPFN_default"
    default_confusion_data = [
        {"model": "TabPFN_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## Tuning across the 5 test folds for one dataset


In [ ]:
def process_parquet_file_tabpfn(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for TabPFN
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Same structure as process_parquet_file_rf in
    07a_tuning_features_ml.ipynb, with tune_and_evaluate_tabpfn_fold as the
    per-fold trainer and n_estimators/outlier_removal_std as the
    hyperparameter columns.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup_all.loc[
        fold_lookup_all["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "behaviour", "fold"],
    ]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(["Ind_ID", "new_burst", "behaviour", "fold"], axis=1)
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        # The N_INNER_FOLDS folds following test_fold are the inner
        # validation folds; each inner round trains on the remaining 3
        # folds (excluding test_fold and that round's validation fold).
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_tabpfn_fold(
            cv_folds,
            X_full[non_test],
            y_enc[non_test],
            X_full[te],
            y_enc[te],
            class_names,
            burst,
            correction_type,
            test_fold,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        # Kept for the post-hoc "Inspect best study per dataset" cell.
        out["studies"][test_fold] = study

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "n_estimators",
        "outlier_removal_std",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    # Default-control metrics: no val_f1_macro/default_val_f1_macro/
    # search_time columns, since no search is run for them.
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "n_estimators",
        "outlier_removal_std",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }

## Run tuning across datasets

In [ ]:
all_results_tabpfn = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_tabpfn(
        pq_file_path, fold_lookup_all, n_trials=N_TRIALS
    )
    all_results_tabpfn.append(result)

multiclass_metrics_tabpfn = pd.concat(
    [r["multiclass_metrics"] for r in all_results_tabpfn], ignore_index=True
)
multiclass_conf_tabpfn = pd.concat(
    [r["multiclass_conf"] for r in all_results_tabpfn], ignore_index=True
)

# Default-hyperparameter control (model="TabPFN_default"), retrained and
# evaluated on the same splits as the tuned winner. Kept separate from
# multiclass_metrics_tabpfn/multiclass_conf_tabpfn (saved to separate files).
default_metrics_tabpfn = pd.concat(
    [r["default_metrics"] for r in all_results_tabpfn], ignore_index=True
)
default_conf_tabpfn = pd.concat(
    [r["default_conf"] for r in all_results_tabpfn], ignore_index=True
)

# Kept in memory (not saved) for the "Inspect best study per dataset"
# cell: all_studies_tabpfn[dataset_key] is a dict of {test_fold: study}.
all_studies_tabpfn = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_tabpfn)
}

## Pool results and inspect the winning hyperparameters per fold

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-fold metrics when classes are unevenly represented across folds.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())


    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def pooled_summary_per_dataset(conf_df, model_name):
    """Pooled metrics computed separately per (burst, correction_type)."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(grp, model_name)
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


# This session's TabPFN results, merged with the rows already on disk in
# the save cell.
multiclass_metrics_run = multiclass_metrics_tabpfn
multiclass_conf_run = multiclass_conf_tabpfn
default_metrics_run = default_metrics_tabpfn
default_conf_run = default_conf_tabpfn

# Models present in this run's results.
MODEL_NAMES = sorted(multiclass_conf_run["model"].unique())
print(f"Models pooled this run: {MODEL_NAMES}")

pooled_multiclass_summary = pd.concat(
    [pooled_summary_per_dataset(multiclass_conf_run, m) for m in MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled tuned multiclass metrics per dataset:")
print(pooled_multiclass_summary)

# Per-fold winning hyperparameters and test metrics. val_f1_macro vs
# default_val_f1_macro compares the winner with the default on the inner
# validation folds.
per_fold_cols = [
    "burst",
    "correction_type",
    "test_fold",
    "val_folds",
    "n_test_samples",
    "model",
    "n_estimators",
    "outlier_removal_std",
    "val_f1_macro",
    "default_val_f1_macro",
    "accuracy",
    "balanced_acc",
    "f1_macro",
    "search_time",
    "time",
]
per_fold_multiclass = multiclass_metrics_run[per_fold_cols].sort_values(
    ["burst", "correction_type", "test_fold"]
)
print(
    "\nPer-fold tuned multiclass results (winning hyperparameters + final test metrics):"
)
print(per_fold_multiclass)

# Number of test folds where the search beat the default on validation,
# and the mean validation gain.
beats_default = (
    per_fold_multiclass["val_f1_macro"] > per_fold_multiclass["default_val_f1_macro"]
)
print(
    f"\nSearch beat the default on validation in {beats_default.sum()}/{len(beats_default)} "
    f"outer folds (mean val_f1_macro - default_val_f1_macro = "
    f"{(per_fold_multiclass['val_f1_macro'] - per_fold_multiclass['default_val_f1_macro']).mean():.4f})"
)

# Test-side default control: pooled tuned vs pooled default metrics on the
# same test folds, and the f1_macro gain.
DEFAULT_MODEL_NAMES = sorted(default_conf_run["model"].unique())
pooled_default_summary = pd.concat(
    [pooled_summary_per_dataset(default_conf_run, m) for m in DEFAULT_MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled DEFAULT (test-fold) metrics per dataset:")
print(pooled_default_summary)

test_gain = pooled_multiclass_summary.assign(
    model_base=pooled_multiclass_summary["model"]
).merge(
    pooled_default_summary.assign(
        model_base=pooled_default_summary["model"].str.replace(
            "_default", "", regex=False
        )
    ),
    on=["burst", "correction_type", "model_base"],
    suffixes=("_tuned", "_default"),
)[["burst", "correction_type", "model_base", "f1_macro_tuned", "f1_macro_default"]]
test_gain["f1_macro_gain"] = test_gain["f1_macro_tuned"] - test_gain["f1_macro_default"]
print("\nTest-set gain (tuned - default, pooled f1_macro):")
print(test_gain)

## Inspect best study per dataset

Run after tuning. For the dataset in `INSPECT_DATASET_TABPFN` (any tuple from
`DATASETS`), picks the test fold with the highest validation `f1_macro`
and shows its Optuna study summary and plots.


In [ ]:
# Dataset to inspect; must be one of the tuples in DATASETS.
INSPECT_DATASET_TABPFN = DATASETS[0]

dataset_rows_tabpfn = multiclass_metrics_tabpfn[
    (multiclass_metrics_tabpfn["burst"].str.lower() == INSPECT_DATASET_TABPFN[0])
    & (multiclass_metrics_tabpfn["correction_type"] == INSPECT_DATASET_TABPFN[1])
]
best_row_tabpfn = dataset_rows_tabpfn.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_tabpfn = best_row_tabpfn["test_fold"]
study_tabpfn = all_studies_tabpfn[INSPECT_DATASET_TABPFN][best_test_fold_tabpfn]

complete_trials_tabpfn = [
    t for t in study_tabpfn.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_TABPFN} "
    f"(by validation f1_macro across its {len(dataset_rows_tabpfn)} outer folds): "
    f"test_fold={best_test_fold_tabpfn}"
)
print(f"  # trials (completed) : {len(complete_trials_tabpfn)}/{len(study_tabpfn.trials)}")
print(f"  Best value            : {study_tabpfn.best_value:.4f}")
print(f"  Best params           : {study_tabpfn.best_params}")
print(f"  Default params        : {TABPFN_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_tabpfn['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_tabpfn.best_value - best_row_tabpfn['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_tabpfn).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Helper: merge new results into existing output CSVs

In [ ]:
def upsert_csv(new_df, path, key_cols):
    """Merge new_df's rows into the CSV at path (creating it if it
    doesn't exist yet), keeping new_df's rows wherever key_cols collide
    with what's already on disk.

    Rows are keyed by key_cols, which include "model", so re-running
    replaces only this notebook's TabPFN rows.
    """
    if os.path.exists(path):
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=key_cols, keep="last")
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return combined

## Save outputs

In [ ]:
# Merges this session's TabPFN results (built in the pooling cell) into
# the output CSVs via upsert_csv. Only the behaviour target is saved.
METRIC_KEY = ["burst", "correction_type", "target", "test_fold", "model"]
CONF_KEY = [
    "burst",
    "correction_type",
    "target",
    "test_fold",
    "model",
    "actual_label",
    "predicted_label",
]

upsert_csv(
    multiclass_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_tuned_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    multiclass_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_tuned_confusion.csv",
    CONF_KEY,
)

# Default-hyperparameter control (test-fold retrain), saved to separate
# files so the tuned metrics/confusion files read by
# 12b_publication_plots_fig3.R contain only tuned rows.
upsert_csv(
    default_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_tuned_default_control_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    default_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_tuned_default_control_confusion.csv",
    CONF_KEY,
)